# Predictive Anayltics: Support Vector Machines

TODO: add embedded, check why there are Nans in lanlong

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [ ]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 2000
SPATIAL_UNIT = "census_tract" # options: census_tract, h3_cell, community_area
SPATIAL_ENCODING = "embedding" # options: embedding, latlong, onehot
MODE = "full" # options: full, sample
TIME_UNIT = "4h" # options: 1h, 2h, 4h
H3_RES = 7 # options 7,8

In [2]:
CENSUS_PATH = "../data/full/raw_data/Census_Tracts.csv"
COMM_PATH = "../data/full/raw_data/Community_Areas.csv"

In [3]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import load, dump
import h3

from shapely import wkt

import geopandas as gpd
from shapely.geometry import Polygon
from srai.neighbourhoods import H3Neighbourhood
from srai.loaders import OSMOnlineLoader
from srai.regionalizers import H3Regionalizer, geocode_to_region_gdf
from srai.joiners import IntersectionJoiner
from srai.h3 import ring_buffer_h3_regions_gdf
from srai.embedders import Hex2VecEmbedder
import pytorch_lightning as py_light

from sklearn.preprocessing import OneHotEncoder

import networkx as nx
from libpysal.weights import Queen
from node2vec import Node2Vec

C:\Users\bkran\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preparations

In [4]:
INPUT = "../data/" + MODE + "/train_test_data/" 

In [ ]:
# Paths, depending on spatial and time unit
DATA_PATH_TRAIN = INPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_train.parquet"
DATA_PATH_TEST = INPUT + "svm_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_test.parquet"
if SPATIAL_UNIT == "h3_cell":
   DATA_PATH_TEST = INPUT + "svm_" + SPATIAL_UNIT + "_" + H3_RES + "_" + TIME_UNIT + "_test.parquet" 
    DATA_PATH_TRAIN = INPUT + "svm_" + SPATIAL_UNIT + "_"+ H3_RES + "_" + TIME_UNIT + "_train.parquet"

MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL, # gets encoded
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
    "h3_cell"
]

Load data and select features and target

In [6]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
test = pl.scan_parquet(DATA_PATH_TEST)

In [7]:
print(train.collect_schema().names())

['datetime_hour', 'month', 'weekday', 'hour', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'hour_sin', 'hour_cos', 'tmpc', 'relh', 'sknt', 'vsby', 'p01m', 'skyc1_BKN', 'skyc1_CLR', 'skyc1_FEW', 'skyc1_OVC', 'skyc1_SCT', 'skyc1_VV ', 'date', 'is_holiday', 'census_tract', 'food_drink', 'landmark', 'shop', 'train_station', 'trip_count', 'trip_seconds_sum', 'trip_seconds_mean', 'trip_seconds_min', 'trip_seconds_max', 'trip_miles_sum', 'trip_miles_mean', 'trip_miles_min', 'trip_miles_max', 'fare_sum', 'fare_mean', 'fare_min', 'fare_max', 'tips_sum', 'tips_mean', 'tips_min', 'tips_max', 'tolls_sum', 'tolls_mean', 'tolls_min', 'tolls_max', 'extras_sum', 'extras_mean', 'extras_min', 'extras_max', 'trip_total_sum', 'trip_total_mean', 'trip_total_min', 'trip_total_max', 'most_common_payment_type']


In [8]:
keep_cols = [
    SPATIAL_UNIT,  # "h3_cell"
    "trip_count",  # target is derived from this
    "month_sin", "month_cos", "weekday_sin", "weekday_cos",
    "hour_sin", "hour_cos",
    "tmpc", "relh", "sknt", "vsby", "p01m",
    "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
    "is_holiday",
    "food_drink", "landmark", "shop", "train_station",
]

train_df = train.select(keep_cols).collect().to_pandas()
test_df = test.select(keep_cols).collect().to_pandas()

In [9]:
train_df.isna().sum()

census_tract          0
trip_count            0
month_sin             0
month_cos             0
weekday_sin           0
weekday_cos           0
hour_sin              0
hour_cos              0
tmpc                  0
relh                  0
sknt                  0
vsby                  0
p01m                  0
skyc1_BKN             0
skyc1_CLR             0
skyc1_FEW             0
skyc1_OVC             0
skyc1_SCT             0
skyc1_VV              0
is_holiday            0
food_drink       332272
landmark         332272
shop             332272
train_station    332272
dtype: int64

In [10]:
train_df.count()

census_tract     3137180
trip_count       3137180
month_sin        3137180
month_cos        3137180
weekday_sin      3137180
weekday_cos      3137180
hour_sin         3137180
hour_cos         3137180
tmpc             3137180
relh             3137180
sknt             3137180
vsby             3137180
p01m             3137180
skyc1_BKN        3137180
skyc1_CLR        3137180
skyc1_FEW        3137180
skyc1_OVC        3137180
skyc1_SCT        3137180
skyc1_VV         3137180
is_holiday       3137180
food_drink       2804908
landmark         2804908
shop             2804908
train_station    2804908
dtype: int64

In [11]:
train_df["food_drink"] = train_df["food_drink"].fillna(0.0)
train_df["landmark"] = train_df["landmark"].fillna(0.0)
train_df["shop"] = train_df["shop"].fillna(0.0)
train_df["train_station"] = train_df["train_station"].fillna(0.0)

test_df["food_drink"] = test_df["food_drink"].fillna(0.0)
test_df["landmark"] = test_df["landmark"].fillna(0.0)
test_df["shop"] = test_df["shop"].fillna(0.0)
test_df["train_station"] = test_df["train_station"].fillna(0.0)


In [12]:
train_df["food_drink"]

0          1.0
1          1.0
2          1.0
3          1.0
4          1.0
          ... 
3137175    0.0
3137176    0.0
3137177    0.0
3137178    0.0
3137179    0.0
Name: food_drink, Length: 3137180, dtype: float64

In [13]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low


# definition for demand cause currently the q1 is 0, q2 is 1 and q3 is 3
# I excluded the zeros since a magority of values are zero
train_p90 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 90)
train_p70 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 70)
train_p50 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 50)
train_p25 = np.percentile(train_df.loc[train_df["trip_count"] > 0, "trip_count"], 25)

train_df["trip_demand"] = 0
train_df.loc[train_df["trip_count"] >= train_p25, "trip_demand"] = 1
train_df.loc[train_df["trip_count"] >= train_p50, "trip_demand"] = 2
train_df.loc[train_df["trip_count"] >= train_p70, "trip_demand"] = 3
train_df.loc[train_df["trip_count"] >= train_p90, "trip_demand"] = 4

test_p90 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 90)
test_p70 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 70)
test_p50 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 50)
test_p25 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 25)

test_df["trip_demand"] = 0
test_df.loc[test_df["trip_count"] >= test_p25, "trip_demand"] = 1
test_df.loc[test_df["trip_count"] >= test_p50, "trip_demand"] = 2
test_df.loc[test_df["trip_count"] >= test_p70, "trip_demand"] = 3
test_df.loc[test_df["trip_count"] >= test_p90, "trip_demand"] = 4


In [14]:
print(test_df.loc[test_df["trip_count"] == 0, "trip_count"].count())
print(test_df.loc[test_df["trip_count"] > 0, "trip_count"].count())

1321172
24716


In [15]:
model = SVC()

In [16]:
test_df.loc[test_df["trip_demand"] != "Low", "trip_demand"]

0          0
1          0
2          0
3          0
4          0
          ..
1345883    0
1345884    0
1345885    0
1345886    0
1345887    0
Name: trip_demand, Length: 1345888, dtype: int64

## Encoding

In [17]:
# encoding
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

### Spatial Encoding: LatLong

In [18]:
if (SPATIAL_ENCODING == "latlong"):
    if SPATIAL_UNIT == "h3_cell":
        print("Encoding: latlong and Unit: hexa")
        for df in (train_df, test_df):
            df["lat"], df["lon"] = zip(*df[SPATIAL_UNIT].map(h3.cell_to_latlng))

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        # create 
        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]
    elif SPATIAL_UNIT == "census_tract":
        print("Encoding: latlong and Unit: census_tract")

        # load census tract
        census_data = pd.read_csv(CENSUS_PATH, dtype={"CENSUS_T_1": str})
        census_data["CENSUS_T_1"] = census_data["CENSUS_T_1"].str.zfill(11)

        tract_centroids = census_data.set_index("CENSUS_T_1")[["TRACT_CE_3", "TRACT_CE_2"]]
        tract_centroids.columns = ["lat", "lon"]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(11)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing = df["lat"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")
            n_missing = df["lon"].isna().sum()
            if n_missing:
                print(f"Warning: {n_missing}/{len(df)} rows failed to match a tract centroid")

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]
    elif SPATIAL_UNIT == "community_area":
        print("Encoding: latlong and Unit: community_area")

        census_data = pd.read_csv(COMM_PATH, dtype={"AREA_NUMBE": str})
        census_data["AREA_NUMBE"] = census_data["AREA_NUMBE"].str.zfill(2)

        census_data["geometry"] = census_data["the_geom"].apply(wkt.loads)
        gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  

        gdf["lon"] = gdf.geometry.centroid.x
        gdf["lat"] = gdf.geometry.centroid.y

        tract_centroids = gdf.set_index("AREA_NUMBE")[["lat", "lon"]]

        for df in (train_df, test_df):
            df[SPATIAL_UNIT] = df[SPATIAL_UNIT].astype(str).str.zfill(2)
            df["lat"] = df[SPATIAL_UNIT].map(tract_centroids["lat"])
            df["lon"] = df[SPATIAL_UNIT].map(tract_centroids["lon"])

            # sanity check, catch silent join failures early
            n_missing_lat = df["lat"].isna().sum()
            n_missing_lon = df["lon"].isna().sum()
            if n_missing_lat or n_missing_lon:
                print(f"Warning: {n_missing_lat} lat / {n_missing_lon} lon rows failed to match a community area centroid")

        X_train = train_df[feature_cols]
        X_test = test_df[feature_cols]

        train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
        X_train_grid = train_df_grid[feature_cols]

### Spatial Encoding: OneHotEncoding

In [19]:
if (SPATIAL_ENCODING == "onehot") & (SPATIAL_UNIT == "community_area"):
    # Community_area is a categorical id, not a numeric quantity, so one-hot encode it
    X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
    X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])
    # Keep the dummy columns before scaling turns X_train into a plain array
    train_columns = X_train.columns

    # Make sure test has the same dummy columns as train (in case a community_area is missing)
    X_test = X_test.reindex(columns=train_columns, fill_value=0)

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = pd.get_dummies(train_df_grid[feature_cols], columns=["community_area"])
    X_train_grid = X_train_grid.reindex(columns=train_columns, fill_value=0)


### Spatial Encoding: Spatial Embedding

In [ ]:
if (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "h3_cell"):

    all_cells = set(train_df["h3_cell"]).union(test_df["h3_cell"])


    regions = []
    for cell in all_cells:
        boundary = h3.cell_to_boundary(cell)

        polygon = Polygon([(lon, lat) for lat, lon in boundary])
        regions.append({"h3_cell": cell, "geometry": polygon})

    regions_gdf = gpd.GeoDataFrame(regions).set_index("h3_cell")

    # Features per H3 cell 
    # Take the mean of POI features per cell 
    poi_cols = ["food_drink", "landmark", "shop", "train_station"]
    features_gdf = train_df.groupby("h3_cell")[poi_cols].mean()

    features_gdf = gpd.GeoDataFrame(
        features_gdf,
        geometry=regions_gdf.loc[features_gdf.index, "geometry"]
    )

    regions_gdf = regions_gdf.rename_axis("region_id")
    features_gdf = features_gdf.rename_axis("feature_id")

    # Maps each region to the features that fall within it
    joiner = IntersectionJoiner()
    joint_gdf = joiner.transform(regions_gdf, features_gdf)

    # fit the embedder
    neighbourhood = H3Neighbourhood(regions_gdf)
    embedder = Hex2VecEmbedder(encoder_sizes=[64, 32])
    embeddings = embedder.fit_transform(
        regions_gdf,
        features_gdf,
        joint_gdf,
        neighbourhood
    )

    # embeddings is a DataFrame indexed by h3_cell with 32 columns

    # Merge embeddings into train/val/test
    emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
    embeddings.columns = emb_cols

    train_df = train_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")
    test_df = test_df.merge(embeddings, left_on="h3_cell", right_index=True, how="left")

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = train_df_grid[feature_cols]
elif (SPATIAL_ENCODING == "embedding") & (SPATIAL_UNIT == "census_tract"):
    census_data = gpd.read_file(CENSUS_PATH)   
    census_data["geometry"] = gpd.GeoSeries.from_wkt(census_data["the_geom"])
    
    gdf = gpd.GeoDataFrame(census_data, geometry="geometry", crs="EPSG:4326")  
    
    gdf["TRACT_FIPS"] = gdf["TRACT_FIPS"].astype(str)
    gdf = gdf.set_index("TRACT_FIPS")
    

    gdf = gdf.to_crs(epsg=5070)
    gdf = gdf[gdf.geometry.notnull() & gdf.geometry.is_valid]

    gdf = gdf[~gdf.index.duplicated(keep="first")]
    w = Queen.from_dataframe(gdf, use_index=True)
    G = w.to_networkx()  
    print(f"Graph: {G.number_of_nodes()} tracts, {G.number_of_edges()} adjacency edges")

    node2vec = Node2Vec(
        G,
        dimensions=32,
        walk_length=20,
        num_walks=100,
        workers=4,
        p=1,
        q=1,
    )
    
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    
    # Build a DataFrame of embeddings indexed by GEOID
    embedding_dict = {node: model.wv[node] for node in G.nodes()}
    emb_df = pd.DataFrame.from_dict(embedding_dict, orient="index")

    train_df["census_tract"] = train_df["census_tract"]
    test_df["census_tract"] = test_df["census_tract"]

    train_df = train_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")
    test_df = test_df.merge(emb_df, left_on="census_tract", right_index=True, how="left")

    X_train = train_df[feature_cols]
    X_test = test_df[feature_cols]

    train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)
    X_train_grid = train_df_grid[feature_cols]



Graph: 1 tracts, 0 adjacency edges


Computing transition probabilities: 100%|██████████| 1/1 [00:00<00:00, 1934.64it/s]


In [ ]:
emb_df.head()

,n2v_0,n2v_1,n2v_2,n2v_3,n2v_4,n2v_5,n2v_6,n2v_7,n2v_8,n2v_9,...,n2v_22,n2v_23,n2v_24,n2v_25,n2v_26,n2v_27,n2v_28,n2v_29,n2v_30,n2v_31
GEOID,,,,,,,,,,,,,,,,,,,,,
0,-0.001676,0.000739,0.015948,0.028154,-0.029072,-0.02224,0.020184,0.028041,-0.015673,-0.011761,...,0.021118,0.002384,0.019847,-0.010642,-0.002958,0.018027,-0.023505,-0.0123,-0.023474,-0.002906


In [22]:
train_df.head()

,census_tract,trip_count,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,...,n2v_22,n2v_23,n2v_24,n2v_25,n2v_26,n2v_27,n2v_28,n2v_29,n2v_30,n2v_31
0,17031560200,0,0.0,1.0,-0.974928,-0.222521,-0.866025,0.5,4.5520,91.7860,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,17031560200,0,0.0,1.0,0.781831,0.623490,0.866025,-0.5,1.3900,90.4850,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,17031560200,0,0.0,1.0,0.781831,0.623490,0.000000,1.0,2.4975,86.1350,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,17031560200,0,0.0,1.0,0.781831,0.623490,0.866025,0.5,3.3300,85.3500,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17031560200,0,0.0,1.0,0.781831,0.623490,-0.866025,0.5,-15.6975,54.6475,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Create y

In [23]:
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]
y_train_grid = train_df_grid[TARGET_COL]

In [24]:
feature_cols

['census_tract',
 'month_sin',
 'month_cos',
 'weekday_sin',
 'weekday_cos',
 'hour_sin',
 'hour_cos',
 'tmpc',
 'relh',
 'sknt',
 'vsby',
 'p01m',
 'skyc1_BKN',
 'skyc1_CLR',
 'skyc1_FEW',
 'skyc1_OVC',
 'skyc1_SCT',
 'skyc1_VV ',
 'is_holiday',
 'food_drink',
 'landmark',
 'shop',
 'train_station']

In [25]:
# scaling since, SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_grid = scaler.fit_transform(X_train_grid)

In [26]:
X_train

array([[ 9.71396830e-01, -1.14660981e-01,  1.30948797e+00, ...,
        -3.05556974e-01, -2.87310415e-01, -2.70137592e-01],
       [ 9.71396830e-01, -1.14660981e-01,  1.30948797e+00, ...,
        -3.05556974e-01, -2.87310415e-01, -2.70137592e-01],
       [ 9.71396830e-01, -1.14660981e-01,  1.30948797e+00, ...,
        -3.05556974e-01, -2.87310415e-01, -2.70137592e-01],
       ...,
       [ 1.52520662e+00, -1.34731902e+00,  5.97735503e-01, ...,
        -3.05556974e-01, -5.68241571e-04, -2.70137592e-01],
       [ 1.52520662e+00, -1.34731902e+00,  5.97735503e-01, ...,
        -3.05556974e-01, -5.68241571e-04, -2.70137592e-01],
       [ 1.52520662e+00, -8.26336434e-01,  1.11877447e+00, ...,
        -3.05556974e-01, -5.68241571e-04, -2.70137592e-01]])

## Grid Search

In [27]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVC(),
        param_grid=grid,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

linear best score: 0.98649999324662 best params: {'C': 0.1, 'kernel': 'linear'}
rbf_sigmoid best score: 0.9879999939969956 best params: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
poly best score: 0.987499493496495 best params: {'C': 0.1, 'degree': 3, 'gamma': 0.01, 'kernel': 'poly'}
Overall best: rbf_sigmoid {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}


In [28]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV score: 0.9879999939969956


In [29]:
#missing_mask = test_df["lat"].isna()
#print(test_df.loc[missing_mask, SPATIAL_UNIT].unique())

In [30]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

#rint("Test accuracy:", accuracy_score(y_test, y_pred))

## Training Model

In [ ]:
# Train SVC 
best_model.fit(X_train, y_train)

## Testing Model

In [ ]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

[[18605   471    96  5921  2521]
 [  959 93086  1727  8517    35]
 [ 1356 30912  1159 12668    23]
 [ 6039 14662   874 24101   129]
 [ 2158     1     1    12 11219]]
              precision    recall  f1-score   support

        High       0.64      0.67      0.66     27614
         Low       0.67      0.89      0.76    104324
         Mid       0.30      0.03      0.05     46118
    Mid High       0.47      0.53      0.50     45805
   Very High       0.81      0.84      0.82     13391

    accuracy                           0.62    237252
   macro avg       0.58      0.59      0.56    237252
weighted avg       0.56      0.62      0.56    237252



## Save Model and Grid Search

In [ ]:
# save model
dump(best_model, "../models/model_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_" + SPATIAL_ENCODING + ".joblib")
dump(grid_search, "../models/grid_" + SPATIAL_UNIT + "_" + TIME_UNIT + "_svc_"+ SPATIAL_ENCODING + ".joblib")

['../models/test/grid_community_svc.joblib']